### Impacto do Tempo de Processamento de Armazém
**Pergunta de Negócio:** O tempo de preparação/processamento do pedido no armazém está associado à ocorrência de atrasos?

**Métricas Calculadas:**
- Quantidade total de pedidos por status de entrega (`qtd_pedidos`).
- Tempo médio de processamento em horas (`media_processamento_hrs`).

In [0]:
%sql
SELECT
  late_delivery AS pedido_atrasado,
  COUNT(*) AS qtd_pedidos,
  ROUND(avg(warehouse_processing_hours), 2) AS media_processamento_hrs
FROM workspace.default.entregas_tratadas
GROUP BY late_delivery
ORDER BY late_delivery;

**Insight / Resultado:**

Os pedidos atrasados apresentaram tempo médio de processamento de 16,59 horas, contra 11,60 horas nos pedidos entregues no prazo. Essa diferença indica uma associação entre o tempo de processamento e a ocorrência de atrasos, sugerindo que a etapa de preparação dos pedidos merece atenção operacional.


###  Relação com Tentativas de Entrega
**Pergunta de Negócio:** Pedidos que exigem múltiplas tentativas de entrega apresentam maior frequência de atraso?

**Métricas Calculadas:**
 Total de pedidos (`total_pedidos`), pedidos atrasados (`pedidos_atrasados`) e taxa de atraso (`percentual_atraso`) por quantidade de tentativas (`tentativas_entrega`).

In [0]:
%sql
SELECT
    delivery_attempts AS tentativas_entrega,
    COUNT(*) AS total_pedidos,
    SUM(CASE
        WHEN late_delivery = 'Yes' THEN 1
        ELSE 0
    END) AS pedidos_atrasados,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY delivery_attempts
ORDER BY delivery_attempts;

**Insight / Resultado:**

A taxa de atraso sobe progressivamente conforme aumentam as tentativas: **55,98%** (1 tentativa), **57,44%** (2 tentativas) e **75,00%** (3 tentativas). Como apenas 292 pedidos exigiram 3 tentativas, o resultado do último grupo deve ser interpretado com cautela devido à amostragem menor, porém evidencia que falhas na entrega presencial aumentam severamente o risco de estouro de prazo.

### Impacto da Distância do Percurso Logístico
**Pergunta de Negócio:** Rotas de maior distância apresentam proporções significativamente maiores de atraso?

**Métricas Calculadas:** 
Agrupamento por faixas de quilometragem (`faixa_distancia`), total de pedidos, volume de atrasos e taxa de atraso percentual.

In [0]:
%sql
-- 1. distancia percorrida e atrasos

SELECT
    CASE
        WHEN distance_km <500 THEN 'Até 499 km'
        WHEN distance_km < 1000 THEN '500 a 999 km'
        WHEN distance_km < 1500 THEN '1000 a 1499 km'
        ELSE '1500 km ou mais'
    END AS faixa_distancia,
    COUNT(*) AS total_pedidos,
    SUM(CASE            
        WHEN late_delivery = 'Yes' THEN 1
        ELSE 0
    END) AS pedidos_atrasados,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2   
    ) AS percentual_atrasos
FROM workspace.default.entregas_tratadas
GROUP BY 
    CASE
        WHEN distance_km <500 THEN 'Até 499 km'
        WHEN distance_km < 1000 THEN '500 a 999 km'
        WHEN distance_km < 1500 THEN '1000 a 1499 km'
        ELSE '1500 km ou mais'
    END
ORDER BY MIN(distance_km)

**Insight / Resultado:**
Existe uma correlação direta entre a distância e o percentual de atraso: rotas curtas (até 499 km) possuem **36,55%** de atraso, enquanto rotas longas (1.500 km ou mais) chegam a **63,89%** de taxa de atraso (uma diferença de **27,34 pontos percentuais**). Isso sugere que a promessa de entrega para rotas longas precisa ser ajustada no sistema.

###  Associação com Condições Climáticas
**Pergunta de Negócio:** A distância percorrida e as condições climáticas estão associadas a atrasos maiores?

**Métricas Calculadas:** 
Mapeamento por condição climática (`condicao_climatica`), contagem de pedidos, volume de atrasos e percentual ordenado de forma decrescente.


In [0]:
%sql
-- 2. condicoes climaticas e atrasos

SELECT
    weather_condition AS condicao_climatica,
    COUNT(*) AS total_pedidos,
    SUM(CASE
        WHEN late_delivery = 'Yes' THEN 1
        ELSE 0
    END) AS pedidos_atrasados,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY weather_condition
ORDER BY percentual_atraso DESC;

**Insight / Resultado:**
Eventos climáticos severos representam o principal gargalo externo: **Neve (93,51%)** e **Tempestades (91,00%)** possuem taxas críticas de atraso. Em contraste, entregas sob tempo bom/céu limpo registram **34,97%** de atraso. A análise reforça a importância de considerar fatores climáticos na estimativa dinâmica de SLA.
